**Notebook version: 9** — Claude will state the version number after editing any cell in this notebook.

# fiftyone_final_dataset.ipynb — browse the FINAL training dataset, all splits at once

Loads `dataset/final/{train,val,test}/` into a **single** FiftyOne dataset so the whole
36,875-image pool is browsable in one App session, with **`source` and `split` as real,
sidebar-filterable fields**.

This is the gap the other notebooks leave. `fiftyone_review_processed.ipynb` accepts
`source_key = "final/train"`, but it browses one split at a time and never sets a `source`
field — in `dataset/final/` the source survives only as the `<source>__` filename prefix
(`merge.py`/`prefixed_filename()`), which the App sidebar cannot filter on.

---

### This notebook is READ-ONLY, deliberately

There is no write-back cell here and there must not be one. `dataset/final/` is a **derived**
directory — `split.py` deletes and regenerates it wholesale on every cascade run, so any box
you fixed here would be silently destroyed the next time the cascade runs.

Corrections belong upstream, in `fiftyone_review_processed.ipynb`, which writes to
`dataset/processed/<source>/labels_reviewed/` and is promoted into `labels/` by
`promote_reviews.py` before the cascade. That is the only edit path that survives.

Use this notebook to **inspect and verify** — per-source class balance, what actually landed
in `val` (which Hailo's DFC uses for calibration), whether a source looks wrong at a glance.
Then go fix it upstream.


In [9]:
# Imports
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Same helper the other notebooks use -- notebooks live in notebooks/, and
    Jupyter's working directory depends on how it was launched.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo

from scripts.utils.config_loader import get_canonical_names
from scripts.utils.file_utils import final_dir, list_images, reports_dir

CANONICAL_NAMES = get_canonical_names()
print(f"repo root      : {REPO_ROOT}")
print(f"canonical nc   : {len(CANONICAL_NAMES)}")
print(f"canonical names: {CANONICAL_NAMES}")


repo root      : /Users/luna/Projects/Thesis/second-vision-ai
canonical nc   : 14
canonical names: ['Person', 'Vehicle', 'Motorcycle', 'Pole', 'Animals', 'Doors', 'Chairs', 'Tables', 'Tricycle', 'Potholes', 'Trash Bins', 'Bicycle', 'Stairs', 'Bench']


In [10]:
# ---- What to load ----

# Which splits to pull into the single browsable dataset. Keep all three for a
# whole-dataset view; narrow it if you only care about one (e.g. ["val"] when
# sanity-checking Hailo's calibration set specifically).
SPLITS = ["train", "val", "test"]

# Optional build-time narrowing. Leave both None to load everything and do all
# your filtering interactively in the App sidebar -- that is the intended
# workflow, and 36,875 images loads fine (labels + paths only, no image decoding).
#
# Set these only when you want a genuinely smaller dataset object, e.g. to hand
# one source to a colleague or to keep an App session snappy on a slow machine.
SOURCE_FILTER = None   # e.g. {"roboflow_pothole_voxrl", "dataset_ninja_pothole_detection"}
CLASS_FILTER = None    # e.g. {"Potholes"} -- keeps images containing >=1 box of these

# Cap per split, for a quick look without waiting on the full build. None = no cap.
MAX_IMAGES_PER_SPLIT = None

# Rebuilding drops and recreates the dataset. Safe here in a way it is NOT in
# fiftyone_review_processed.ipynb: this dataset holds no review work -- every
# field is re-derived from dataset/final/ on disk in about a minute.
DATASET_NAME = "final_dataset_browse"

# ---- Which dataset TREE to browse (DEC-126 / DEC-127) ----
#
# v1 and v2 live side by side on purpose; v1 is never overwritten.
#
#   "final"      -> dataset/final/      15 classes, original labels        (v1)
#   "final_v2a"  -> dataset/final_v2a/  15 classes, recovered boxes, Shelf still present
#   "final_v2b"  -> dataset/final_v2b/  14 classes, Shelf dropped + renumbered  <-- the DEC-126 gate
#
# NOTE the trees differ in nc, so the class NAMES differ too. final_v2b has no
# "Shelf" and every id above 4 shifted down by one -- which is exactly what the
# visual gate exists to confirm, so do not "fix" a surprising label by editing
# this; check it in the App.
DATASET_TREE = "final_v2b"

_TREE_ROOT = REPO_ROOT / "dataset" / DATASET_TREE
assert _TREE_ROOT.is_dir(), f"no such tree: {_TREE_ROOT}"


def tree_dir(split: str | None = None):
    """final_dir() but honouring DATASET_TREE. Same return shape."""
    return _TREE_ROOT / split if split else _TREE_ROOT


print(f"browsing tree  : {_TREE_ROOT}")


browsing tree  : /Users/luna/Projects/Thesis/second-vision-ai/dataset/final_v2b


In [11]:
# ---- Build one FiftyOne dataset spanning every requested split ----
#
# `source` and `split` are stored as top-level sample fields, which is the whole
# point of this notebook: both become sidebar filters in the App. The source is
# recovered from merge.py's filename prefix (prefixed_filename(): "<source>__<name>"),
# the same convention box_audit.py --pool merged already relies on (DEC-075).

def _source_of(filename: str) -> str:
    """Recover the source key from a merged/final filename prefix.

    merge.py prefixes every file it writes, so a name with no "__" means the
    pool was not built by merge.py -- surfaced rather than silently bucketed.
    """
    if "__" not in filename:
        return "<unprefixed>"
    return filename.split("__", 1)[0]


# Integrity check before trusting anything below: config/classes.yaml is the
# authoritative schema, dataset/final/data.yaml is what training actually reads.
# They are generated to agree (generate_yaml.py, DEC-065) -- if they have drifted,
# every count in this notebook is being labelled with the wrong names.
# DEC-126 made this a WARNING rather than a hard error. Three trees now coexist
# with DIFFERENT schemas on purpose -- final (15c), final_v2a (15c), final_v2b (14c)
# -- so disagreement with the live config is expected whenever you browse an older
# tree, and erroring out would make v1 unbrowsable. The safety intent is preserved
# by labelling with the TREE'S OWN names (TREE_NAMES) rather than the live config:
# a tree is always displayed with the schema its labels were written under, which is
# exactly what stops the DEC-107 "every Pothole shows as Tricycle" failure.
data_yaml_path = tree_dir() / "data.yaml"
if data_yaml_path.is_file():
    import yaml
    _dy = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
    TREE_NAMES = [_dy["names"][i] for i in sorted(_dy["names"])]
    if TREE_NAMES != CANONICAL_NAMES:
        print(
            f"NOTE: '{DATASET_TREE}' uses a DIFFERENT schema from the live config.\n"
            f"  this tree     : nc={len(TREE_NAMES)} {TREE_NAMES}\n"
            f"  classes.yaml  : nc={len(CANONICAL_NAMES)} {CANONICAL_NAMES}\n"
            f"  -> labelling with THIS TREE'S names, which is correct for browsing it.\n"
        )
    else:
        print(f"data.yaml <-> classes.yaml: agree at nc={len(TREE_NAMES)}")
else:
    TREE_NAMES = list(CANONICAL_NAMES)
    print(f"NOTE: {data_yaml_path} not found -- falling back to config/classes.yaml names.")

if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)
    print(f"Dropped existing '{DATASET_NAME}' (re-derived from disk, no review work lost).")

dataset = fo.Dataset(DATASET_NAME)
# DEC-081: without this the App blocks on an "Import your dataset schema" prompt.
dataset.classes["ground_truth"] = list(TREE_NAMES)

samples = []
skipped_missing_label = 0
skipped_by_filter = 0
unknown_class_ids: Counter = Counter()

for split in SPLITS:
    images_dir = tree_dir(split) / "images"
    labels_dir = tree_dir(split) / "labels"
    if not images_dir.is_dir():
        print(f"  {split}: {images_dir} not found -- skipped.")
        continue

    image_paths = sorted(list_images(images_dir, recursive=False))
    if MAX_IMAGES_PER_SPLIT is not None:
        image_paths = image_paths[:MAX_IMAGES_PER_SPLIT]

    kept = 0
    for image_path in image_paths:
        source = _source_of(image_path.name)
        if SOURCE_FILTER is not None and source not in SOURCE_FILTER:
            skipped_by_filter += 1
            continue

        label_path = labels_dir / f"{image_path.stem}.txt"
        if not label_path.is_file():
            # Tracked, not silently dropped -- same posture as merge.py/split.py.
            skipped_missing_label += 1
            continue

        detections = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:])
            if not (0 <= class_id < len(TREE_NAMES)):
                unknown_class_ids[class_id] += 1
                continue
            # YOLO centre-relative -> FiftyOne top-left-relative.
            detections.append(
                fo.Detection(
                    label=TREE_NAMES[class_id],
                    bounding_box=[cx - w / 2, cy - h / 2, w, h],
                )
            )

        if CLASS_FILTER is not None and not any(d.label in CLASS_FILTER for d in detections):
            skipped_by_filter += 1
            continue

        sample = fo.Sample(filepath=str(image_path))
        sample["source"] = source
        sample["split"] = split
        sample["num_boxes"] = len(detections)
        sample["ground_truth"] = fo.Detections(detections=detections)
        samples.append(sample)
        kept += 1

    print(f"  {split}: {kept} samples")

dataset.add_samples(samples)
dataset.persistent = False  # a viewer, not review state -- regenerated in ~1 min

# Index the fields the App sidebar filters on. NOT an optimisation -- a
# correctness fix, and the reason this cell must re-run it after every rebuild.
#
# FiftyOne 1.20's sidebar searches UNINDEXED fields with a bounded scan and, when
# it runs out of budget, returns a PARTIAL value list with a small "Incomplete
# search. create an index" note under the dropdown. It does not fail or blank out,
# it just quietly shows fewer values than exist -- so the sidebar looked like the
# dataset was missing classes and sources when nothing was missing at all.
#
# Measured here on 2026-09-05, before indexing: `source` offered 5 of its 12 real
# values (every roboflow_* source invisible) and `ground_truth.label` offered 12 of
# 13 (Doors missing). What gets dropped is NOT predictable and differs by field --
# `source` kept the 5 alphabetically-first values, while `ground_truth.label` dropped
# Doors from the MIDDLE of the alphabet (it is the rarest class, 2,280 boxes). So a
# truncated list can look like a complete one; the only reliable tell is the
# "Incomplete search" note. Both lists are complete once indexed.
for _field in ("source", "split", "ground_truth.detections.label"):
    dataset.create_index(_field)
print("Sidebar indexes created (source, split, ground_truth.detections.label) -- "
      "without these the App's filter dropdowns silently show a PARTIAL value list.")

print(f"\nLoaded {len(dataset)} samples across {len(SPLITS)} split(s).")
if skipped_missing_label:
    print(f"WARNING: {skipped_missing_label} image(s) had no matching label file.")
if skipped_by_filter:
    print(f"{skipped_by_filter} image(s) excluded by SOURCE_FILTER/CLASS_FILTER.")
if unknown_class_ids:
    print(f"WARNING: out-of-range class ids found (schema drift?): {dict(unknown_class_ids)}")


data.yaml <-> classes.yaml: agree at nc=14
Dropped existing 'final_dataset_browse' (re-derived from disk, no review work lost).
  train: 30123 samples
  val: 6644 samples
  test: 6403 samples
 100% |█████████████| 43170/43170 [16.2s elapsed, 0s remaining, 2.6K samples/s]      
Sidebar indexes created (source, split, ground_truth.detections.label) -- without these the App's filter dropdowns silently show a PARTIAL value list.

Loaded 43170 samples across 3 split(s).


In [12]:
# ---- Side-by-side: what did v2 actually ADD? (DEC-127) ----
#
# You do NOT need a second App window. This overlays the OLD tree's labels onto the
# SAME samples as a second field, and tags EVERY INDIVIDUAL BOX that v2 added or
# dropped, so you can filter down to just the new boxes instead of just the images.
#
# After running this, in the App:
#   * LABEL TAGS sidebar -> `new_box`     : ONLY the boxes v2 added   <-- the new-box filter
#                        -> `removed_box` : ONLY the boxes v2 dropped (on ground_truth_v1)
#   * SAMPLE TAGS        -> `gained_boxes` / `lost_boxes` : jump to the images that changed
#   * Labels section     -> tick/untick `ground_truth` vs `ground_truth_v1` to flip
#     between the new and old boxes on the image you are looking at
#
# Matching is GEOMETRIC (same label AND IoU >= MATCH_IOU), never string equality —
# float round-tripping through disk makes string comparison of boxes unreliable.
#
# Set to None to skip the overlay entirely.
COMPARE_AGAINST = "final"        # "final" (v1) | "final_v2a" | None
MATCH_IOU = 0.9                  # >= this, same label => the same box

def _iou(a, b):
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    if x2 <= x1 or y2 <= y1:
        return 0.0
    inter = (x2 - x1) * (y2 - y1)
    return inter / (aw * ah + bw * bh - inter)

if COMPARE_AGAINST and COMPARE_AGAINST != DATASET_TREE:
    import yaml as _yaml
    _old_root = REPO_ROOT / "dataset" / COMPARE_AGAINST
    _oy = _yaml.safe_load((_old_root / "data.yaml").read_text(encoding="utf-8"))
    OLD_NAMES = [_oy["names"][i] for i in sorted(_oy["names"])]
    print(f"overlaying {COMPARE_AGAINST} (nc={len(OLD_NAMES)}) onto {DATASET_TREE} "
          f"(nc={len(TREE_NAMES)}) as field 'ground_truth_v1'")

    dataset.classes["ground_truth_v1"] = list(OLD_NAMES)
    n_gained = n_lost = n_same = n_absent = 0
    n_new_boxes = n_removed_boxes = 0

    for smp in dataset.iter_samples(autosave=True, progress=True):
        stem = Path(smp.filepath).stem
        old_lp = _old_root / smp["split"] / "labels" / f"{stem}.txt"

        # clear any tags from a previous run so re-running is idempotent
        smp.tags = [t for t in smp.tags
                    if t not in ("gained_boxes", "lost_boxes", "not_in_" + COMPARE_AGAINST)]
        cur = smp["ground_truth"].detections if smp["ground_truth"] else []
        for d in cur:
            d.tags = [t for t in (d.tags or []) if t != "new_box"]

        if not old_lp.is_file():
            smp.tags = smp.tags + ["not_in_" + COMPARE_AGAINST]
            n_absent += 1
            continue

        dets = []
        for line in old_lp.read_text().split("\n"):
            line = line.strip()
            if not line:
                continue
            cid, cx, cy, w, h = line.split()[:5]
            cid, cx, cy, w, h = int(cid), float(cx), float(cy), float(w), float(h)
            if not (0 <= cid < len(OLD_NAMES)):
                continue
            dets.append(fo.Detection(label=OLD_NAMES[cid],
                                     bounding_box=[cx - w / 2, cy - h / 2, w, h]))

        # greedy one-to-one geometric matching, new tree vs old tree
        old_taken = [False] * len(dets)
        gained = lost = 0
        for d in cur:
            best, best_i = 0.0, -1
            for i, o in enumerate(dets):
                if old_taken[i] or o.label != d.label:
                    continue
                v = _iou(d.bounding_box, o.bounding_box)
                if v > best:
                    best, best_i = v, i
            if best >= MATCH_IOU:
                old_taken[best_i] = True
            else:
                d.tags = (d.tags or []) + ["new_box"]
                gained += 1
        for i, o in enumerate(dets):
            if not old_taken[i]:
                o.tags = (o.tags or []) + ["removed_box"]
                lost += 1

        smp["ground_truth_v1"] = fo.Detections(detections=dets)
        n_new_boxes += gained
        n_removed_boxes += lost
        if gained and not lost:
            smp.tags = smp.tags + ["gained_boxes"]; n_gained += 1
        elif lost and not gained:
            smp.tags = smp.tags + ["lost_boxes"]; n_lost += 1
        elif gained and lost:
            smp.tags = smp.tags + ["gained_boxes", "lost_boxes"]; n_gained += 1; n_lost += 1
        else:
            n_same += 1

    print(f"\n  new boxes     : {n_new_boxes:,}  (label tag `new_box` on ground_truth)")
    print(f"  removed boxes : {n_removed_boxes:,}  (label tag `removed_box` on ground_truth_v1)")
    print(f"\n  gained_boxes : {n_gained:,} images  <-- filter to these in the App")
    print(f"  lost_boxes   : {n_lost:,} images  (expected: Shelf-only and DEC-115 Bench drops)")
    print(f"  unchanged    : {n_same:,} images")
    if n_absent:
        print(f"  not in {COMPARE_AGAINST}: {n_absent:,} images")

    for _f in ("tags",
               "ground_truth.detections.tags",
               "ground_truth_v1.detections.label",
               "ground_truth_v1.detections.tags"):
        try:
            dataset.create_index(_f)
        except Exception:
            pass

    print("\n  Suggested checks:")
    print("    LABEL TAG new_box + source=open_images + label=Chairs -> the tables-folder recovery")
    print("    LABEL TAG new_box + source=crowdhuman                 -> Chairs/Tables/Bicycle, NOT Doors/Trash Bins")
    print("    LABEL TAG new_box + source=exdark + label=Vehicle     -> the 413 Bus boxes (DEC-129)")
    print("    label=Potholes                                        -> must be road damage, NOT tricycles")
    print("    label=Shelf                                           -> must return ZERO in final_v2b")
else:
    print("COMPARE_AGAINST disabled or same as DATASET_TREE -- no overlay added.")


overlaying final (nc=15) onto final_v2b (nc=14) as field 'ground_truth_v1'
 100% |█████████████| 43170/43170 [39.9s elapsed, 0s remaining, 1.1K samples/s]      

  new boxes     : 35,468  (label tag `new_box` on ground_truth)
  removed boxes : 10,242  (label tag `removed_box` on ground_truth_v1)

  gained_boxes : 10,216 images  <-- filter to these in the App
  lost_boxes   : 2,889 images  (expected: Shelf-only and DEC-115 Bench drops)
  unchanged    : 31,558 images

  Suggested checks:
    LABEL TAG new_box + source=open_images + label=Chairs -> the tables-folder recovery
    LABEL TAG new_box + source=crowdhuman                 -> Chairs/Tables/Bicycle, NOT Doors/Trash Bins
    LABEL TAG new_box + source=exdark + label=Vehicle     -> the 413 Bus boxes (DEC-129)
    label=Potholes                                        -> must be road damage, NOT tricycles
    label=Shelf                                           -> must return ZERO in final_v2b


In [ ]:
# ---- Inspect the HIERARCHY-DUPLICATE proposal (DEC-133) — changes NO label ----
#
# The 0.90 geometric dedup (DEC-132) was too strict: duplicate annotations of the
# same object survive down to ~0.70. But a plain 0.70 threshold is the WRONG fix —
# it would delete 131 genuinely distinct adjacent objects (bicycles in a rack,
# chairs in a row, people in a crowd).
#
# The real discriminator is the RAW Open Images native label. DEC-127 collapsed
# hierarchy parents/children/siblings onto one canonical class:
#     Person <- Person, Man, Woman, Boy, Girl
#     Tables <- Table, Desk, Kitchen & dining room table, Coffee table
#     Chairs <- Chair, Couch, Stool, Sofa bed
# Open Images often annotates ONE object under several at once — Desk+Table 229x,
# Girl+Woman 161x, Man+Person 72x. After mapping that is two boxes on one object.
#
#     DIFFERENT native labels + IoU >= 0.70  ->  one object, two names  -> DROP one
#     SAME native label       + IoU >= 0.70  ->  two real objects       -> KEEP both
#
# Generate/refresh the proposal first (also writes no labels):
#     python3 scripts/preprocess/find_hierarchy_duplicates.py --iou 0.70
#
# In the App, under LABEL TAGS:
#     `dup_would_drop` - the 728 boxes that would be REMOVED
#     `dup_kept`       - the twin each one is a duplicate of
#     `dup_protected`  - the 131 SAME-native pairs the rule deliberately KEEPS.
#                        ** Check these first. ** If they look like two real
#                        objects, the rule is right. If they look like one object
#                        with two boxes, tell me and the rule needs revisiting.
#
# Click any flagged box to read `dup_iou` and `dup_native`.

REPORT = REPO_ROOT / "dataset" / "reports" / "dup_hierarchy_candidates.json"

_TAGS = ("dup_pair", "dup_would_drop", "dup_kept", "dup_protected",
         "dup_hierarchy", "dup_v1_plus_new", "dup_both_new", "dup_both_v1")

if not REPORT.is_file():
    print(f"  {REPORT} not found — run:")
    print("     python3 scripts/preprocess/find_hierarchy_duplicates.py --iou 0.70")
else:
    import json as _json
    from collections import Counter, defaultdict
    _rep = _json.loads(REPORT.read_text())

    def _to_xywh(line):
        f = line.split()
        cx, cy, w, h = map(float, f[1:5])
        return (cx - w / 2, cy - h / 2, w, h)

    def _same(a, b, tol=1e-6):
        return all(abs(x - y) <= tol for x, y in zip(a, b))

    want = defaultdict(list)
    for c in _rep["candidates"]:
        want[c["stem"]].append(("drop", c))
        want[c["stem"]].append(("keep", c))
    for c in _rep.get("protected", []):
        want[c["stem"]].append(("prot", c))

    n_drop = n_keep = n_prot = n_missed = 0
    _cls = Counter()

    for smp in dataset.iter_samples(autosave=True, progress=True):
        stem = Path(smp.filepath).stem
        base = stem.split("__", 1)[1] if "__" in stem else stem
        gt = smp["ground_truth"]
        dets = gt.detections if (gt and gt.detections) else []
        for d in dets:
            d.tags = [t for t in (d.tags or []) if t not in _TAGS]
        if base not in want:
            continue
        for role, c in want[base]:
            if role == "prot":
                for key in ("line_a", "line_b"):
                    tgt = _to_xywh(c[key])
                    for d in dets:
                        if d.label == c["canonical"] and _same(tuple(d.bounding_box), tgt):
                            d.tags = list(set(d.tags or []) | {"dup_pair", "dup_protected"})
                            d["dup_iou"] = c["iou"]
                            d["dup_native"] = f"both {c['native']} — two real objects, KEPT"
                            n_prot += 1
                            break
                continue
            key = "line_dropped" if role == "drop" else "line_kept"
            tgt = _to_xywh(c[key])
            hit = None
            for d in dets:
                if d.label == c["canonical"] and _same(tuple(d.bounding_box), tgt):
                    hit = d
                    break
            if hit is None:
                if role == "drop":
                    n_missed += 1
                continue
            if role == "drop":
                hit.tags = list(set(hit.tags or []) | {"dup_pair", "dup_would_drop", "dup_hierarchy"})
                hit["dup_native"] = f"{c['native_dropped']} (DROP) vs {c['native_kept']} (keep)"
                n_drop += 1
                _cls[c["canonical"]] += 1
            else:
                hit.tags = list(set(hit.tags or []) | {"dup_pair", "dup_kept", "dup_hierarchy"})
                hit["dup_native"] = f"{c['native_kept']} (keep) vs {c['native_dropped']} (DROP)"
                n_keep += 1
            hit["dup_iou"] = c["iou"]

    print(f"\n  proposal IoU threshold : {_rep['iou_threshold']}")
    print(f"  tagged dup_would_drop  : {n_drop:,}  (report says {_rep['n_boxes_to_drop']:,})")
    print(f"  tagged dup_kept        : {n_keep:,}")
    print(f"  tagged dup_protected   : {n_prot:,}  ({_rep['same_native_pairs_protected']:,} pairs)")
    if n_missed:
        print(f"  !! {n_missed} candidates not matched to a detection — investigate before applying")
    print(f"\n  would-drop by class: {dict(_cls.most_common())}")
    print("\n  by native-label pair:")
    for k, v in list(_rep["by_native_pair"].items())[:10]:
        print(f"     {k:52s} {v:5,}")

    try:
        dataset.create_index("ground_truth.detections.tags")
    except Exception:
        pass

    print("""
  CHECK dup_protected FIRST — those are the ones the rule spares.
  Then spot-check dup_would_drop: each should sit on top of a dup_kept twin.
  Nothing on disk has changed by running this cell.""")


In [14]:
# ---- Cross-check against split_report.json ----
#
# Confirms this notebook is looking at the same pool split.py actually wrote,
# rather than a stale or partially-copied dataset/final/. Only meaningful on a
# full, unfiltered build.

report_path = reports_dir() / "split_report.json"
if not report_path.is_file():
    print("split_report.json not found -- skipping cross-check.")
elif SOURCE_FILTER or CLASS_FILTER or MAX_IMAGES_PER_SPLIT:
    print("Filters active -- cross-check skipped (counts intentionally differ).")
else:
    report = json.loads(report_path.read_text(encoding="utf-8"))
    expected = report["split_counts"]
    actual = Counter(s["split"] for s in dataset.select_fields("split"))
    ok = True
    for split in SPLITS:
        exp, act = expected.get(split), actual.get(split, 0)
        flag = "OK" if exp == act else "<-- MISMATCH"
        if exp != act:
            ok = False
        print(f"  {split:<6} report={exp:<7} loaded={act:<7} {flag}")
    leakage = report.get("cross_split_duplicate_leakage")
    print(f"\ncross_split_duplicate_leakage: {leakage if leakage else '[] (none)'}")
    print("Counts match split_report.json." if ok else
          "MISMATCH -- dataset/final/ is not what split.py last wrote. Re-run the cascade.")


  train  report=30123   loaded=30123   OK
  val    report=6645    loaded=6644    <-- MISMATCH
  test   report=6403    loaded=6403    OK

cross_split_duplicate_leakage: [] (none)
MISMATCH -- dataset/final/ is not what split.py last wrote. Re-run the cascade.


In [15]:
# ---- Per-source x per-split breakdown ----
#
# The table the sidebar cannot give you at a glance. Sorted by total images.

by_source_split: dict[str, Counter] = defaultdict(Counter)
by_source_class: dict[str, Counter] = defaultdict(Counter)
by_class_split: dict[str, Counter] = defaultdict(Counter)

for sample in dataset.select_fields(["source", "split", "ground_truth"]):
    src, split = sample["source"], sample["split"]
    by_source_split[src][split] += 1
    by_source_split[src]["total"] += 1
    seen = {d.label for d in (sample.ground_truth.detections if sample.ground_truth else [])}
    for label in seen:
        by_source_class[src][label] += 1
        by_class_split[label][split] += 1

print("IMAGES BY SOURCE x SPLIT")
print(f"{'source':<42} {'train':>7} {'val':>7} {'test':>7} {'total':>8}")
print("-" * 75)
for src, c in sorted(by_source_split.items(), key=lambda kv: -kv[1]["total"]):
    print(f"{src:<42} {c['train']:>7} {c['val']:>7} {c['test']:>7} {c['total']:>8}")
print("-" * 75)
tot = Counter()
for c in by_source_split.values():
    tot.update(c)
print(f"{'TOTAL':<42} {tot['train']:>7} {tot['val']:>7} {tot['test']:>7} {tot['total']:>8}")

print("\n\nIMAGES BY CLASS x SPLIT   (an image counts once per class it contains)")
print(f"{'class':<14} {'train':>7} {'val':>7} {'test':>7} {'total':>8}   {'val%':>6} {'test%':>6}")
print("-" * 75)
for name in CANONICAL_NAMES:
    c = by_class_split.get(name, Counter())
    t = c["train"] + c["val"] + c["test"]
    vp = f"{c['val']/t:.1%}" if t else "-"
    tp = f"{c['test']/t:.1%}" if t else "-"
    print(f"{name:<14} {c['train']:>7} {c['val']:>7} {c['test']:>7} {t:>8}   {vp:>6} {tp:>6}")


IMAGES BY SOURCE x SPLIT
source                                       train     val    test    total
---------------------------------------------------------------------------
open_images                                  16753    3619    3581    23953
roboflow_dlsu_d_vehicle_type_detection        4308     926     911     6145
exdark                                        3878     917     840     5635
roboflow_revised_pedestrian_obstacle          1054     226     225     1505
dataset_ninja_road_damage_detector             928     205     197     1330
roboflow_cv_project_hovyc                      878     196     188     1262
roboflow_pothole_voxrl                         495     106      64      665
dataset_ninja_pothole_detection                407     163      94      664
roboflow_door_detection_zqt59                  449      96      97      642
roboflow_trashcan_detection_pihfn              407      69      83      559
roboflow_roitrikee                             302      65     

In [16]:
# ---- Which classes does each source actually contribute? ----
#
# Worth a look because several sources are no longer single-class: review passes
# added correct boxes for other classes visible in the same photo (DEC-087), so a
# "pothole source" may legitimately carry Person and Vehicle boxes too.

print("CLASSES PER SOURCE (image counts)")
for src, c in sorted(by_source_class.items(), key=lambda kv: -sum(kv[1].values())):
    parts = ", ".join(f"{k}={v}" for k, v in c.most_common())
    print(f"\n  {src}")
    print(f"    {parts}")


CLASSES PER SOURCE (image counts)

  open_images
    Person=8128, Tables=4771, Bench=3245, Animals=3025, Bicycle=2966, Chairs=2816, Stairs=2630, Pole=2182, Motorcycle=1659, Vehicle=1561, Trash Bins=1104

  roboflow_dlsu_d_vehicle_type_detection
    Vehicle=4184, Motorcycle=2310, Person=1657, Tricycle=1546, Bicycle=117, Chairs=36, Animals=15, Tables=3, Pole=2

  exdark
    Person=2658, Animals=1608, Chairs=1230, Vehicle=1082, Tables=995, Bicycle=751, Motorcycle=587

  dataset_ninja_road_damage_detector
    Potholes=1330, Motorcycle=1135, Person=1125, Vehicle=977, Bicycle=41, Chairs=19, Pole=13, Animals=3, Tricycle=2, Tables=2, Doors=1

  roboflow_revised_pedestrian_obstacle
    Person=1227, Vehicle=648, Motorcycle=471, Stairs=297, Tricycle=285, Pole=221, Chairs=117, Bicycle=78, Doors=36, Tables=31, Animals=21, Trash Bins=18

  roboflow_cv_project_hovyc
    Doors=1256, Person=99, Stairs=77, Chairs=38, Vehicle=37, Pole=32, Trash Bins=20, Bicycle=17, Tables=11, Animals=5, Motorcycle=3

  r

## Filtering in the App

Once the App is open, both fields you asked for are in the **left sidebar**:

- **`source`** — every source key (`open_images`, `crowdhuman`, `roboflow_pothole_voxrl`, …).
  Click one to see only that source's images. Multi-select works.
- **`split`** — `train` / `val` / `test`.
- **`ground_truth.label`** — filter by class.
- **`num_boxes`** — a range slider; useful for finding empty or absurdly crowded images.

These compose, so "`source = roboflow_pothole_voxrl` AND `split = val`" is two clicks.

For anything the sidebar can't express, use a view in code — e.g.:

```python
from fiftyone import ViewField as F

view = dataset.match(F("source") == "crowdhuman").match(F("num_boxes") > 20)
session.view = view
```


In [17]:
# Launch the App. If a server is already running (it survives kernel restarts),
# this attaches to it rather than starting a second one.
session = fo.launch_app(dataset, auto=False)
session


Session launched. Run `session.show()` to open the App in a cell output.


Dataset:          final_dataset_browse
Media type:       image
Num samples:      43170
Selected samples: 0
Selected labels:  0
Session URL:      http://localhost:5151/

In [18]:
# Run when finished. The dataset is non-persistent, so it also disappears on
# kernel shutdown -- rebuild from the cells above whenever you want it back.
fo.close_app()
print("App closed.")


Exception ignored in: <function Service.__del__ at 0x11c5f2fc0>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/second-vision/lib/python3.11/site-packages/fiftyone/core/service.py", line 86, in __del__
    self.stop()
  File "/opt/anaconda3/envs/second-vision/lib/python3.11/site-packages/fiftyone/core/service.py", line 127, in stop
    self.child.stdin.close()
    ^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'stdin'


App closed.
